# 01 — Problématique et données

**Objectif du projet NutriHFr** : décrire les consommations alimentaires et les apports nutritionnels des adultes en France à partir de l'étude **INCA 3** (Anses, 2014-2015).

## 1. Question scientifique

> Comment se caractérisent les apports nutritionnels des adultes en France, comment varient-ils selon l'âge et le sexe, et quels aliments y contribuent le plus ?

| | Question | Exemple de résultat |
|---|---|---|
| Q1 | Quelle est la distribution de l'apport énergétique journalier ? | moyenne pondérée, IC 95 %, histogramme |
| Q2 | Les apports diffèrent-ils entre hommes et femmes ? | moyennes par sexe |
| Q3 | Comment évoluent-ils avec l'âge ? | moyennes par classe d'âge, régression |
| Q4 | À énergie égale, qui a l'alimentation la plus riche en nutriments ? | apports pour 1 000 kcal |
| Q5 | Quels aliments apportent le plus de protéines, calcium, fer, fibres, vitamine C ? | % de contribution par groupe |
| Q6 | Les apports varient-ils beaucoup d'une personne à l'autre ? | percentiles, violin plot |

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # pour importer le code de src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nutrihfr import config as C

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})
C.DOSSIER_FIGURES.mkdir(parents=True, exist_ok=True)
C.DOSSIER_TABLES.mkdir(parents=True, exist_ok=True)

## 2. Les données disponibles

Il existe plusieurs enquêtes nationales de consommation en France (INCA 1, INCA 2, INCA 3, cohorte NutriNet-Santé). On utilise **uniquement INCA 3** : c'est la plus récente en accès libre, et ses fichiers contiennent à la fois les consommations détaillées, la composition nutritionnelle de chaque aliment consommé et les apports calculés par l'Anses (ce qui permet de valider notre calcul).

Source : [data.gouv.fr — INCA 3](https://www.data.gouv.fr/datasets/donnees-de-consommations-et-habitudes-alimentaires-de-letude-inca-3) · documentation : notice utilisateurs (dans `data/raw/inca3/`).

Fichiers utilisés :

| Fichier | Contenu | Une ligne = |
|---|---|---|
| `description-indiv.csv` | caractéristiques des participants, pondérations | un individu |
| `conso-compo-alim-…csv` | aliments consommés pendant les rappels de 24 h + composition pour 100 g | un aliment consommé |
| `apports-nut-alim.csv` | apports journaliers calculés par l'Anses | un individu |

## 3. Volumétrie et qualité globale des fichiers

In [2]:
from nutrihfr.pipeline import ChargeurINCA3
chargeur = ChargeurINCA3()

lignes = []
for cle, nom in C.FICHIERS.items():
    df = chargeur.lire(cle)
    lignes.append({"fichier": nom, "lignes": len(df), "colonnes": df.shape[1],
                   "taille_Mo": round((C.DOSSIER_BRUT / nom).stat().st_size / 1e6, 1),
                   "% cellules vides": round(100 * df.isna().mean().mean(), 1)})
pd.DataFrame(lignes)

,fichier,lignes,colonnes,taille_Mo,% cellules vides
0,description-indiv.csv,5855,185,2.60,45.70
1,conso-compo-alim-vf-mad-datagouv2021.csv,256301,109,204.60,1.40
2,apports-nut-alim.csv,4114,108,4.40,0.10


Les cellules vides sont nombreuses dans `description-indiv` (questions posées seulement à certains participants) et dans `conso-compo-alim` (facettes descriptives facultatives). Les variables que nous utilisons sont, elles, presque complètes (voir notebook 03).

## 4. Choix de la population

INCA 3 fournit plusieurs pondérations selon le niveau de participation (notice, § 1.5). On étudie les consommations, donc on prend les **adultes « participants CONSO » (Pop3)** et la pondération `pond_indiv_adu_pop3`.

In [3]:
indiv_brut = chargeur.lire("individus")
adultes = indiv_brut[indiv_brut["tage_PS"].isin(C.CLASSES_AGE)]
ids_conso = set(chargeur.lire("consommations")["NOIND"])

recap = []
for pond in ["pond_indiv_adu_pop1", "pond_indiv_adu_pop2", "pond_indiv_adu_pop3"]:
    avec_poids = adultes[adultes[pond].notna()]
    recap.append({"pondération": pond, "adultes concernés": len(avec_poids),
                  "dont avec rappels 24 h": avec_poids["NOIND"].isin(ids_conso).sum(),
                  "population représentée": round(avec_poids[pond].sum())})
pd.DataFrame(recap)

,pondération,adultes concernés,dont avec rappels 24 h,population représentée
0,pond_indiv_adu_pop1,3157,2121,45382836
1,pond_indiv_adu_pop2,2288,2053,45382937
2,pond_indiv_adu_pop3,2121,2121,44941366


`pop3` couvre exactement les 2 121 adultes ayant des rappels de 24 h et représente environ 45 millions d'adultes. C'est notre population d'étude.

## 5. Fiche d'identification

La fiche complète du jeu de données est dans `docs/fiche_donnees.md`. En résumé :

| | |
|---|---|
| Producteur | Anses |
| Période de recueil | 2014-2015 |
| Méthode | 2 ou 3 rappels de 24 h non consécutifs |
| Population étudiée | 2 121 adultes de 18-79 ans, France métropolitaine |
| Clé individu | `NOIND` |
| Confidentialité | données pseudonymisées (pas de nom, âge en classes) |

➡️ Suite : `02_data_engineering.ipynb`